# Polynomial Regression

Polynomial Regression is an extension of Linear Regression that models non-linear relationships between the input features and the target variable. By transforming the original features into polynomial features (powers and cross-terms), it allows a linear model to fit curved patterns in the data. Despite the non-linear transformation, the model remains **linear in its parameters** — making it a special case of multiple linear regression applied to engineered features.

## 1. Key Concepts

Given a training set $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$, standard Linear Regression fits the model:

$$\hat{y} = w_1 x + b$$

This is limited to straight-line relationships. Polynomial Regression overcomes this by first applying a **feature map** $\phi$ that expands $x$ into higher-degree terms:

$$\phi(x) = [x, x^2, x^3, \dots, x^d]$$

Then fitting the linear model on the transformed features:

$$\hat{y} = w_1 x + w_2 x^2 + \cdots + w_d x^d + b = \mathbf{w}^\top \phi(x) + b$$

Where $d$ is the polynomial **degree** and $\mathbf{w} = [w_1, w_2, \dots, w_d]$ are the learned weights.

The key insight is that **the model is non-linear in $x$ but linear in $\mathbf{w}$** — so all standard linear regression solvers (least squares, gradient descent) apply without modification.

## 2. Polynomial Feature Expansion

### Single feature

For a single input feature $x$, the expansion up to degree $d$ is straightforward:

$$\phi(x) = [x,\ x^2,\ x^3,\ \dots,\ x^d]$$

### Multiple features — cross-terms

For a multi-dimensional input $\mathbf{x} = [x_1, x_2, \dots, x_p]$, the expansion also includes **interaction terms** — products of different feature combinations up to degree $d$. For example, with two features $x_1, x_2$ and degree 2:

$$\phi(x_1, x_2) = [x_1,\ x_2,\ x_1^2,\ x_1 x_2,\ x_2^2]$$

The total number of features after expansion with $p$ original features and degree $d$ is:

$$\binom{p + d}{d} - 1$$

**Note**: The implementation uses `combinations_with_replacement` to enumerate all valid monomial terms systematically, ensuring no cross-term is missed.

## 3. Parameter Estimation

Once the polynomial features matrix $\mathbf{X}_{\text{poly}}$ is built, the model delegates parameter estimation to the underlying `LinearRegression` class. Two methods are supported:

### Least Squares (closed-form)

Minimizes the residual sum of squares analytically using the **Normal Equation**:

$$\mathbf{w}^* = (\mathbf{X}_{\text{poly}}^\top \mathbf{X}_{\text{poly}})^{-1} \mathbf{X}_{\text{poly}}^\top \mathbf{y}$$

Fast and exact, but requires the matrix $(\mathbf{X}^\top \mathbf{X})$ to be invertible and can be expensive for very large feature sets.

### Gradient Descent (iterative)

Updates the weights iteratively by following the negative gradient of the Mean Squared Error:

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \nabla_{\mathbf{w}} \mathcal{L}$$

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

Where $\alpha$ is the **learning rate** and the process runs for a fixed number of **epochs**. More flexible for large datasets, but sensitive to the choice of $\alpha$.

## 4. Pseudo-algorithm

**Training phase** — `fit(X, y)`:

1. $\mathcal{D} \leftarrow$ training set $\{(\mathbf{x}_i, y_i)\}_{i=1}^N$, degree $d$, method $m$
2. **for each** sample $\mathbf{x}_i$ **do**
3. $\quad \phi(\mathbf{x}_i) \leftarrow$ all monomials of $\mathbf{x}_i$ up to degree $d$ (including cross-terms)
4. **end for**
5. $\mathbf{X}_{\text{poly}} \leftarrow [\phi(\mathbf{x}_1),\ \phi(\mathbf{x}_2),\ \dots,\ \phi(\mathbf{x}_N)]$
6. $(\mathbf{w}^*, b^*) \leftarrow \text{LinearRegression.fit}(\mathbf{X}_{\text{poly}},\ \mathbf{y},\ \text{method}=m)$
7. **return** $(\mathbf{w}^*, b^*)$

**Prediction phase** — `predict(X)`:

8. **for each** sample $\mathbf{x}_i$ **do**
9. $\quad \phi(\mathbf{x}_i) \leftarrow$ polynomial feature expansion of $\mathbf{x}_i$
10. $\quad \hat{y}_i \leftarrow \mathbf{w}^{*\top} \phi(\mathbf{x}_i) + b^*$
11. **end for**
12. **return** $[\hat{y}_1, \hat{y}_2, \dots, \hat{y}_N]$

## 5. The Degree Hyperparameter

The polynomial degree $d$ is the most critical hyperparameter — it directly controls the **bias-variance trade-off**:

| Degree | Behavior | Risk |
|--------|----------|------|
| Too low (e.g. $d=1$) | Model is too simple, cannot capture non-linear patterns | **Underfitting** (high bias) |
| Just right | Model captures the true underlying relationship | Good generalization |
| Too high (e.g. $d \geq 10$) | Model fits noise in the training set, wild oscillations | **Overfitting** (high variance) |

The optimal degree is typically found through **cross-validation**: fit models with increasing $d$ and select the one that minimizes validation error.

## 6. Implementation

For this implementation, we use the **Diabetes** dataset from Scikit-learn. It contains 442 patient samples with 10 physiological features (age, BMI, blood pressure, etc.) and a continuous target: a **quantitative measure of disease progression** one year after baseline.

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter

diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target)

In [ ]:
X.head()

In [ ]:
# Split the data into training and testing sets
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)

### With Scikit-learn

In [ ]:
import time

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression as SklearnLR
from sklearn.pipeline import make_pipeline

start_1 = time.perf_counter()
sk_poly = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), SklearnLR())
sk_poly.fit(X_train, y_train)
end_1 = time.perf_counter()

### With ifri-mini-ml-lib

In [ ]:
from ifri_mini_ml_lib.regression import PolynomialRegression

start_2 = time.perf_counter()
my_poly = PolynomialRegression(degree=2, method='least_squares')
my_poly.fit(X_train.values, y_train.values)
end_2 = time.perf_counter()

In [ ]:
# Predict on the test set
y_pred_1 = sk_poly.predict(X_test)
y_pred_2 = my_poly.predict(X_test.values)

In [ ]:
# Evaluate both models
from ifri_mini_ml_lib.metrics.regression import mean_squared_error, mean_absolute_error, r2_score

mse_1 = mean_squared_error(y_test, y_pred_1)
mse_2 = mean_squared_error(y_test, y_pred_2)

mae_1 = mean_absolute_error(y_test, y_pred_1)
mae_2 = mean_absolute_error(y_test, y_pred_2)

r2_1  = r2_score(y_test, y_pred_1)
r2_2  = r2_score(y_test, y_pred_2)

In [ ]:
# Summary table
results = pd.DataFrame({
    'Metric': ['MSE', 'MAE', 'R² Score', 'Training Time (s)'],
    'Scikit-learn': [mse_1, mae_1, r2_1, end_1 - start_1],
    'ifri-mini-ml-lib': [mse_2, mae_2, r2_2, end_2 - start_2],
})

results.T

## 7. Interactive Demo

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown
from notebooks.regression.utils import plot_polynomial_regression

interact(
    plot_polynomial_regression,
    degree=IntSlider(min=1, max=10, step=1, value=2, description='Degree:'),
    method=Dropdown(
        options=['least_squares', 'gradient_descent'],
        value='least_squares',
        description='Method:'
    ),
    library=Dropdown(
        options=['sklearn', 'ifri_mini_ml_lib'],
        value='sklearn',
        description='Library:'
    )
);

The interactive demo above allows you to observe the effect of the polynomial degree on the fitted curve. Notice how low degrees produce smooth but inaccurate fits (underfitting), while high degrees produce curves that pass through training points but oscillate wildly (overfitting) — a phenomenon known as **Runge's effect**.

## 8. Real-life Applications

Polynomial Regression is especially useful when the relationship between variables is known or suspected to follow a curved pattern.

1. **Physics and Engineering**: Many physical laws are naturally polynomial — kinetic energy grows with the square of velocity, braking distance grows with the square of speed. Polynomial regression is used to fit sensor data and calibrate physical models.

2. **Epidemiology and Growth Modeling**: Disease spread or population growth can follow non-linear trajectories. Polynomial models are used to capture the shape of growth curves over time before more complex models are applied.

3. **Economics and Demand Forecasting**: The relationship between price and demand is rarely linear. Polynomial regression helps model diminishing returns, saturation effects, and other non-linear economic phenomena.

4. **Computer Graphics and Curve Fitting**: Polynomial regression underpins curve-fitting techniques used in animation, font rendering, and geometric modeling (e.g., Bézier curves are polynomial by construction).

Beyond standalone use, polynomial feature expansion is a foundational technique in **kernel methods** and **feature engineering pipelines**, where it serves as a building block for more expressive models.

## 9. Limitations and Challenges

Polynomial Regression is powerful but comes with practical caveats that must be carefully managed.

1. **Feature Explosion**: With $p$ input features and degree $d$, the number of polynomial features grows as $\binom{p+d}{d} - 1$. For example, 10 features at degree 3 produces 285 terms — making training expensive and increasing the risk of overfitting.

2. **Overfitting at High Degrees**: High-degree polynomials fit training data very closely but generalize poorly to new data. Regularization techniques (Ridge, Lasso) are typically needed to control model complexity.

3. **Sensitivity to Outliers**: Because polynomial features amplify the magnitude of large values, outliers can have a disproportionate influence on the fitted curve, leading to unstable predictions at the boundaries of the input range.

4. **Extrapolation Instability**: Polynomial models behave poorly outside the training data range. High-degree polynomials in particular can produce extreme predictions (very large positive or negative values) for inputs slightly beyond the observed domain.

These limitations motivate the use of regularized variants (Ridge/Lasso regression on polynomial features) or alternative non-linear models (splines, kernel regression, neural networks) when polynomial regression proves insufficient.

## 10. References

- Polynomial and Spline Interpolation, Scikit-learn User Guide, https://scikit-learn.org/stable/modules/linear_model.html#polynomial-regression
- James, G. et al. (2013). *An Introduction to Statistical Learning*. Springer. Chapter 7: Moving Beyond Linearity.
- Polynomial Regression, Towards Data Science, https://towardsdatascience.com/polynomial-regression-bbe8b9d97491
- Bias-Variance Tradeoff, StatQuest with Josh Starmer, https://www.youtube.com/watch?v=EuBBz3bI-aA